In [1]:
import pandas as pd
import numpy as np
import random
import math

In [ ]:
recipes_df = pd.read_csv("./data/recipes.csv")
def getmeals(recipes_df):
    meals = {}
    for _, row in recipes_df.iterrows():
       meals.update({ 
          row["Name"]:(
          row["type"],
          row["Category"],
          row["total_price"],
          row["provided_calories"]
       )})
    return meals

def gettransitionModel(recipes_df):
    result = {}

    for _, row in recipes_df.iterrows():
        meal_type = row["type"]
        name = row["Name"]

         if meal_type not in result:
            result[meal_type] = []

         result[meal_type].append(name)

    return result

class GAProblem:
    def __init__(self, recipes_df, total_price, tdee):
        self.meals = getmeals(recipes_df)
        self.size = 90
        self.total_price = total_price
        self.tdee = tdee
        self.transitionmodel = gettransitionModel(recipes_df)
    
    def get_total_cost(self, state):
        total_cost = 0
        for i in range(len(state)):
           breakfast_cost =  self.meals[state[i][0]][2]
           lunch_cost = self.meals[state[i][1]][2]
           dinner_cost  = self.meals[state[i][2]][2]
           day_cost = breakfast_cost+lunch_cost+dinner_cost
           total_cost += day_cost
        return total_cost

    def get_delta_nutritional(self, state):
       delta_dict = {}
       for i in range(len(state)):
           breakfast_c =  self.meals[state[i][0]][3]
           lunch_c = self.meals[state[i][1]][3]
           dinner_c  = self.meals[state[i][2]][3]
           day_c = breakfast_c+lunch_c+dinner_c
           delta_dict[state[i]] = abs(day_c-self.tdee)
       return delta_dict
    def fitness_function(self,state):
       return 1
    def generate_random_state(self):
         return [(random.choice(self.transitionmodel['Breakfast']),
                  random.choice(self.transitionmodel['Lunch']),
                  random.choice(self.transitionmodel['Dinner']))
                    for _ in range(30)]
def crossover_and_mutation(population,problem):
    cutoff = random.randrange(30)
    childs = []
    for i in range(len(population) - 1):
        for j in range(i+1):
           parent_a = population[i]
           parent_b = population[j]
           child_a = parent_a[:cutoff] + parent_b[cutoff:]
           child_b = parent_b[:cutoff] + parent_a[cutoff:]
           random_lunch_a = random.choice(problem.transitionmodel['Lunch'])
           random_lunch_b = random.choice(problem.transitionmodel['Lunch'])
           random_pos_a  = random.randrange(30)
           random_pos_b  = random.randrange(30)
           child_a[random_pos_a][1] = random_lunch_a # problem we cannot change the tuple
           child_a[random_pos_b][1] = random_lunch_b # problem we cannot change the tuple
           childs.append(child_a)
           childs.append(child_b)
    return childs

def GASearch(problem):
    # step 1 Selection:
    list_of_states = []
    for i in range(30):
     state = problem.generate_random_state()
     fitness = problem.fitness_function(state)
     list_of_states.append((state,fitness))
    
    
    # step 2 and 3 crossover and mutation
    while True: # limit the while so we can choose the best one not the perfect
         population = sorted(list_of_states, key=lambda x: x[1], reverse=True)[:4]
         if any(fitness == 0 for state, fitness in population):
            return state
         population =  crossover_and_mutation(population,problem)

    